# DNA Methylation Gene DEG Heatmap
---
- 读取 DEG Excel 数据，筛选 DNA 甲基化相关基因
- 使用 ComplexHeatmap (R) 画热图
- 行 = 基因, 列 = Model|CellType, 值 = log2FC 或 nlogp
- 列注释 = Model + Region, 行注释 = Category

In [14]:
set.seed(123)

library(tidyr)
library(dplyr)
library(stringr)
library(ComplexHeatmap)
library(circlize)
library(readxl)
library(grDevices)
library(grid)

In [15]:
# ============================================================================
# 配置参数
# ============================================================================

DEG_FILE <- "/data2st2/junyi/output/1-1 six_dataset_DEG_list_fdr_FC01_filtered_7region_rmMB 20251227.xlsx"
DEG_SHEET <- "all"

# 筛选: NULL = 不过滤
FILTER_MODEL   <- NULL       # e.g. "CUSUS M"
FILTER_REGIONS <- NULL       # e.g. c("AMY","HPF","PFC")

# 值类型: "log2FC" 或 "nlogp"
VALUE_TYPE <- "log2FC"

CLUSTER_ROWS <- TRUE
CLUSTER_COLS <- TRUE

In [16]:
# ============================================================================
# DNA 甲基化基因字典
# ============================================================================

dna_methylation_dict <- list(
  "Writers"    = c("Dnmt1", "Dnmt3a", "Dnmt3b"),
  "Co-factors" = c("Dnmt3l", "Uhrf1", "Uhrf2"),
  "Readers"    = c("Mecp2", "Mbd1", "Mbd2", "Mbd3", "Mbd4"),
  "Erasers"    = c("Tet1", "Tet2", "Tet3"),
  "BER_Repair" = c("Tdg", "Neil1", "Neil2", "Smug1", "Mutyh", "Mbd4")
)

gene_category <- stack(dna_methylation_dict)
colnames(gene_category) <- c("Gene", "Category")
all_methylation_genes <- toupper(unique(gene_category$Gene))

cat(sprintf("Genes to search: %d\n", length(all_methylation_genes)))
print(gene_category)

Genes to search: 19
     Gene   Category
1   Dnmt1    Writers
2  Dnmt3a    Writers
3  Dnmt3b    Writers
4  Dnmt3l Co-factors
5   Uhrf1 Co-factors
6   Uhrf2 Co-factors
7   Mecp2    Readers
8    Mbd1    Readers
9    Mbd2    Readers
10   Mbd3    Readers
11   Mbd4    Readers
12   Tet1    Erasers
13   Tet2    Erasers
14   Tet3    Erasers
15    Tdg BER_Repair
16  Neil1 BER_Repair
17  Neil2 BER_Repair
18  Smug1 BER_Repair
19  Mutyh BER_Repair
20   Mbd4 BER_Repair


In [17]:
# ============================================================================
# 读取 DEG 数据
# ============================================================================

df_deg_all <- read_excel(DEG_FILE, sheet = DEG_SHEET)
df_deg_all <- as.data.frame(df_deg_all, stringsAsFactors = FALSE)

cat(sprintf("Loaded %d rows, %d columns\n", nrow(df_deg_all), ncol(df_deg_all)))
cat(sprintf("Columns: %s\n", paste(colnames(df_deg_all), collapse = ", ")))

# 查看有哪些 Model
cat(sprintf("\nUnique Models: %s\n", paste(unique(df_deg_all$Model), collapse = ", ")))
head(df_deg_all, 3)

Loaded 456588 rows, 25 columns
Columns: Gene, pval, log2FC, ci.hi, ci.lo, FDR, Bonferroni, Subclass, Region, Sex, Method, Ensemble, Gene_name, Direction, Neurotransmitter, Region subclass, scores, log2FC_Wilcoxon, pval_Wilcoxon, FDR_Wilcoxon, pct_nz_group, comparison_group, Bonferroni_Wilcoxon, Subclass_Wilcoxon, Model

Unique Models: CUSUS F, CUSUS M, CURES M, CURES F, CSSUS M, CSRES M


,Gene,pval,log2FC,ci.hi,ci.lo,FDR,Bonferroni,Subclass,Region,Sex,⋯,Region subclass,scores,log2FC_Wilcoxon,pval_Wilcoxon,FDR_Wilcoxon,pct_nz_group,comparison_group,Bonferroni_Wilcoxon,Subclass_Wilcoxon,Model
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,⋯,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>,<chr>,<chr>
1,3110056K07Rik,6.631587e-06,-0.1106034,-0.05770798,-0.1634989,0.0002334871,0.02241477,Astrocyte-2,STR,F,⋯,STR Astrocyte-2,-3.997054,-0.6779382,6.413565e-05,0.0139567716,0.1201581,SUS,1.00000000,Astrocyte-2,CUSUS F
2,9330159F19Rik,6.545594e-05,-0.1870986,-0.10210552,-0.2720917,0.0015471405,0.22124109,Astrocyte-2,STR,F,⋯,STR Astrocyte-2,-4.810071,-0.4459102,1.508771e-06,0.0007695203,0.4339921,SUS,0.02462465,Astrocyte-2,CUSUS F
3,AC149090.1,3.904284e-06,-0.1690630,-0.09228789,-0.2458381,0.0001466275,0.01319648,Astrocyte-2,STR,F,⋯,STR Astrocyte-2,-4.826543,-0.5451636,1.389235e-06,0.0007314096,0.2964427,SUS,0.02267370,Astrocyte-2,CUSUS F


In [18]:
# ============================================================================
# 筛选甲基化基因
# ============================================================================

# 确保有 Gene 列
if (!("Gene" %in% colnames(df_deg_all))) {
  first_col <- colnames(df_deg_all)[1]
  df_deg_all$Gene <- df_deg_all[[first_col]]
}

df_deg_all$Gene_upper <- toupper(df_deg_all$Gene)
df_meth <- df_deg_all[df_deg_all$Gene_upper %in% all_methylation_genes, ]

cat(sprintf("Found %d rows for methylation genes\n", nrow(df_meth)))

found_genes <- unique(df_meth$Gene_upper)
cat(sprintf("Matched (%d): %s\n", length(found_genes), paste(found_genes, collapse = ", ")))
missing_genes <- setdiff(all_methylation_genes, found_genes)
if (length(missing_genes) > 0) {
  cat(sprintf("Missing (%d): %s\n", length(missing_genes), paste(missing_genes, collapse = ", ")))
}

Found 446 rows for methylation genes
Matched (8): DNMT3A, TET3, TET1, TET2, UHRF2, MBD1, MECP2, MBD2
Missing (11): DNMT1, DNMT3B, DNMT3L, UHRF1, MBD3, MBD4, TDG, NEIL1, NEIL2, SMUG1, MUTYH


In [19]:
# 可选过滤
if (!is.null(FILTER_MODEL)) {
  df_meth <- df_meth[grepl(FILTER_MODEL, df_meth$Model, fixed = TRUE), ]
  cat(sprintf("Filter Model='%s': %d rows\n", FILTER_MODEL, nrow(df_meth)))
}
if (!is.null(FILTER_REGIONS)) {
  df_meth <- df_meth[df_meth$Region %in% FILTER_REGIONS, ]
  cat(sprintf("Filter Regions: %d rows\n", nrow(df_meth)))
}

# 添加 Category
df_meth <- merge(df_meth, gene_category, by.x = "Gene_upper", by.y = "Gene", all.x = TRUE)

In [20]:
# ============================================================================
# 创建列名: sample = Model|Region_Subclass
# ============================================================================

# 清理 Model 名
df_meth$Model_clean <- gsub(" ", "_", df_meth$Model)

# 创建 cell type 名
if (all(c("Region", "Subclass") %in% colnames(df_meth))) {
  df_meth$ctname <- paste(df_meth$Region, df_meth$Subclass, sep = "_")
  df_meth$ctname <- gsub("/", "-", df_meth$ctname)
  df_meth$ctname <- gsub(" ", "_", df_meth$ctname)
} else {
  df_meth$ctname <- df_meth[["Region Subclass"]]
  df_meth$ctname <- gsub("/", "-", df_meth$ctname)
  df_meth$ctname <- gsub(" ", "_", df_meth$ctname)
}

# sample = Model | celltype
df_meth$sample <- paste(df_meth$Model_clean, df_meth$ctname, sep = "|")
df_meth$term_name <- df_meth$Gene

cat(sprintf("Unique samples (Model|CellType): %d\n", length(unique(df_meth$sample))))
cat(sprintf("Unique Models: %s\n", paste(unique(df_meth$Model_clean), collapse = ", ")))
head(df_meth[, c("Gene", "Category", "Model_clean", "sample", "log2FC", "FDR")], 5)

Unique samples (Model|CellType): 283
Unique Models: CUSUS_F, CUSUS_M, CURES_M, CURES_F, CSSUS_M, CSRES_M


,Gene,Category,Model_clean,sample,log2FC,FDR
,<chr>,<fct>,<chr>,<chr>,<dbl>,<dbl>
1,Dnmt3a,NA,CUSUS_F,CUSUS_F|TH_TH_Fam20a_Fbln1_Glut,0.1459031,4.225678e-08
2,Dnmt3a,NA,CUSUS_F,CUSUS_F|TH_TH_Fgf10_Gm16263_Glut,0.2260031,4.320716e-08
3,Dnmt3a,NA,CUSUS_F,CUSUS_F|TH_TH_Kcnk13_Nox4_Glut,0.1777098,2.158633e-07
4,Dnmt3a,NA,CUSUS_F,CUSUS_F|TH_TH_Meis2_Gm2115_GABA,0.1212526,6.615275e-03
5,Dnmt3a,NA,CUSUS_F,CUSUS_F|HY_HY_Lama1_Gpr149_Glut,0.1287681,1.533176e-06


In [21]:
# ============================================================================
# 计算 value
# ============================================================================

if (VALUE_TYPE == "nlogp") {
  df_meth$FDR_num <- suppressWarnings(as.numeric(as.character(df_meth$FDR)))
  df_meth$log2FC_num <- suppressWarnings(as.numeric(as.character(df_meth$log2FC)))
  df_meth$FDR_num[!is.na(df_meth$FDR_num) & df_meth$FDR_num == 0] <- 1e-300
  df_meth$value <- -log10(df_meth$FDR_num) * sign(df_meth$log2FC_num)
  df_meth$value[is.na(df_meth$value)] <- 0
  legend_name <- "nlogp"
} else {
  df_meth$value <- suppressWarnings(as.numeric(as.character(df_meth$log2FC)))
  df_meth$value[is.na(df_meth$value)] <- 0
  legend_name <- "log2FC"
}

# 去重聚合
df_meth_agg <- df_meth %>%
  group_by(sample, term_name, Gene_upper, Category) %>%
  summarise(
    value  = mean(value, na.rm = TRUE),
    Region = dplyr::first(Region),
    Model  = dplyr::first(Model_clean),
    .groups = "drop"
  )

cat(sprintf("After aggregation: %d rows\n", nrow(df_meth_agg)))

After aggregation: 446 rows


In [22]:
# ============================================================================
# 构建矩阵: 行=基因, 列=Model|CellType
# ============================================================================

heatmap_data <- df_meth_agg %>%
  select(sample, term_name, value) %>%
  pivot_wider(names_from = term_name, values_from = value, values_fn = mean)

mat <- as.data.frame(heatmap_data)
rownames(mat) <- mat$sample
mat$sample <- NULL
mat <- as.matrix(mat)
mat[is.na(mat)] <- 0

mat_t <- t(mat)  # rows=genes, cols=Model|CellType

cat(sprintf("Matrix: %d genes x %d columns\n", nrow(mat_t), ncol(mat_t)))
cat(sprintf("Value range: [%.3f, %.3f]\n", min(mat_t), max(mat_t)))
print(round(mat_t, 3))

Matrix: 8 genes x 283 columns
Value range: [-0.458, 0.771]


       CSRES_M|AMY_AMY_Ccdc3_Acvr1c_Glut CSRES_M|AMY_AMY_Crhbp_Maf_GABA
Tet2                              -0.128                         -0.159
Dnmt3a                             0.000                          0.000
Mecp2                              0.000                          0.000
Tet1                               0.000                          0.000
Tet3                               0.000                          0.000
Mbd2                               0.000                          0.000
Uhrf2                              0.000                          0.000
Mbd1                               0.000                          0.000
       CSRES_M|AMY_AMY_Fign_Ostm1_GABA CSRES_M|AMY_AMY_Foxp2_Penk_GABA
Tet2                            -0.117                          -0.258
Dnmt3a                           0.000                           0.000
Mecp2                            0.000                           0.000
Tet1                             0.000                           0.0

In [23]:
# ============================================================================
# 列注释 (Model + Region) & 行注释 (Category)
# ============================================================================

# 列元数据
sample_meta <- df_meth_agg %>%
  select(sample, Region, Model) %>%
  distinct()

col_anno_df <- data.frame(sample = colnames(mat_t), stringsAsFactors = FALSE)
col_anno_df <- merge(col_anno_df, sample_meta, by = "sample", all.x = TRUE)
rownames(col_anno_df) <- col_anno_df$sample
col_anno_df$sample <- NULL

# 行元数据
gene_to_cat <- df_meth_agg %>%
  select(term_name, Category) %>%
  distinct()
colnames(gene_to_cat)[1] <- "gene"

row_anno_df <- data.frame(gene = rownames(mat_t), stringsAsFactors = FALSE)
row_anno_df <- merge(row_anno_df, gene_to_cat, by = "gene", all.x = TRUE)
rownames(row_anno_df) <- row_anno_df$gene
row_anno_df$gene <- NULL

cat("Column annotations:\n"); print(head(col_anno_df))
cat("\nRow annotations:\n"); print(row_anno_df)

Column annotations:
                                  Region   Model
CSRES_M|AMY_AMY_Ccdc3_Acvr1c_Glut    AMY CSRES_M
CSRES_M|AMY_AMY_Crhbp_Maf_GABA       AMY CSRES_M
CSRES_M|AMY_AMY_Fign_Ostm1_GABA      AMY CSRES_M
CSRES_M|AMY_AMY_Foxp2_Penk_GABA      AMY CSRES_M
CSRES_M|AMY_AMY_Rai14_Foxp2_GABA     AMY CSRES_M
CSRES_M|AMY_AMY_Rorb_Smoc1_Glut      AMY CSRES_M

Row annotations:
       Category
Dnmt3a     <NA>
Mbd1       <NA>
Mbd2       <NA>
Mecp2      <NA>
Tet1       <NA>
Tet2       <NA>
Tet3       <NA>
Uhrf2      <NA>


In [24]:
# ============================================================================
# 颜色定义
# ============================================================================

base_region <- c(
  AMY="#ff7f0e", STR="#e377c2", PFC="#8c564b", iCTX="#1f77b4",
  MB="#9467bd", TH="#bcbd22", HY="#d62728", HPF="#009E73"
)

base_model <- c(
  "CUSUS_M"  = "#00B0F0", "CUSUS_F"  = "#A5D86E",
  "CSSUS_M"  = "#2B6CB8", "CURES_M"  = "#E63CE6",
  "CURES_F"  = "#F6B1C3", "CSRES_M"  = "#BD551C",
  "SUS"      = "#46A6B2", "RES"      = "#DB6B94",
  "CUMS"     = "#A88ED3", "CSDS"     = "#7A4A6F"
)

base_category <- c(
  "Writers"    = "#E41A1C", "Co-factors" = "#377EB8",
  "Readers"    = "#4DAF4A", "Erasers"    = "#984EA3",
  "BER_Repair" = "#FF7F00"
)

make_palette <- function(values, base_map) {
  vals <- as.character(unique(values))
  vals <- vals[!is.na(vals)]
  base_sub <- base_map[intersect(names(base_map), vals)]
  missing <- setdiff(vals, names(base_map))
  if (length(missing) > 0) {
    extra <- grDevices::hcl(
      h = seq(15, 375, length.out = length(missing) + 1)[-1], c = 70, l = 65)
    names(extra) <- missing
    base_sub <- c(base_sub, extra[missing])
  }
  base_sub[match(vals, names(base_sub))]
}

dyn_colors <- list()
if ("Region" %in% colnames(col_anno_df)) {
  dyn_colors$Region <- make_palette(col_anno_df$Region, base_region)
}
if ("Model" %in% colnames(col_anno_df)) {
  dyn_colors$Model <- make_palette(col_anno_df$Model, base_model)
}
if ("Category" %in% colnames(row_anno_df)) {
  dyn_colors$Category <- make_palette(row_anno_df$Category, base_category)
}

In [25]:
# ============================================================================
# 画热图
# ============================================================================

# 热图颜色: 蓝-白-红
rng <- range(mat_t, finite = TRUE)
m_val <- max(abs(rng))
at <- seq(-m_val, m_val, length.out = 5)
col_fun <- circlize::colorRamp2(
  c(min(at), 0, max(at)), c("#3498db", "#FFFFFF", "#d62728"))

# 列注释 (Model + Region) & 列分割 (Model)
col_ha <- HeatmapAnnotation(
  Model  = col_anno_df$Model,
  Region = col_anno_df$Region,
  col = dyn_colors,
  annotation_name_side = "right",
  annotation_name_gp = gpar(fontsize = 10, fontface = "bold")
)
col_split <- col_anno_df$Model

# 行注释 (Category) & 行分割
row_split <- row_anno_df$Category
row_ha <- rowAnnotation(
  Category = row_anno_df$Category,
  col = dyn_colors,
  annotation_name_side = "top",
  annotation_name_gp = gpar(fontsize = 10, fontface = "bold")
)

# 尺寸 (宽窄版)
plot_height <- max(6, nrow(mat_t) / 3 + 4)
plot_width  <- max(6, ncol(mat_t) / 5 + 4)

dir.create("figures", showWarnings = FALSE)
output_pdf <- sprintf("figures/methylation_%s.pdf", VALUE_TYPE)
pdf(output_pdf, height = plot_height, width = plot_width)

ht <- Heatmap(
  mat_t,
  name = legend_name,
  col = col_fun,
  heatmap_legend_param = list(
    at = at, labels = format(at, digits = 2),
    title = legend_name,
    title_gp = gpar(fontsize = 10, fontface = "bold"),
    labels_gp = gpar(fontsize = 8)),
  show_row_names     = TRUE,
  show_column_names  = TRUE,
  cluster_rows       = CLUSTER_ROWS,
  cluster_columns    = CLUSTER_COLS,
  cluster_column_slices = FALSE,
  row_split          = row_split,
  column_split       = col_split,
  left_annotation    = row_ha,
  top_annotation     = col_ha,
  row_names_side     = "right",
  column_names_side  = "bottom",
  row_names_gp       = gpar(fontsize = 10),
  column_names_gp    = gpar(fontsize = 6),
  use_raster         = FALSE,
  row_title          = NULL,
  column_title       = sprintf("DNA Methylation Genes - %s", VALUE_TYPE)
)

draw(ht)
dev.off()
cat(sprintf("\nSaved: %s\n", output_pdf))

pdf 
  2


Saved: figures/methylation_log2FC.pdf


In [26]:
# ============================================================================
# 快捷: 直接 source 独立 R 脚本
# ============================================================================

# source("draw_methylation_heatmap.R")
#
# 修改参数后再 source:
# VALUE_TYPE <- "nlogp"
# FILTER_MODEL <- "CUSUS M"
# FILTER_REGIONS <- c("AMY", "HPF", "PFC")
# source("draw_methylation_heatmap.R")